## Imports

In [1]:
import json
import re
import time
import warnings
import logging
import xlsxwriter

import folium
import pandas as pd
import geopandas as gpd
import numpy as np
import requests
from rosreestr2coord import Area
from shapely.geometry import Polygon, Point
from folium.plugins import MarkerCluster


warnings.filterwarnings('ignore')

## Loading railwas stations

In [3]:
esr_df = pd.read_csv(r"data\esr.csv", sep=';')
osm2esr_df = pd.read_csv(r"data\osm2esr.csv", sep=';')

moscow_railway_stations = esr_df[esr_df['source'] == 'mosobl'][['division', 'railway', 'type', 'esr']].merge(
    osm2esr_df[osm2esr_df['status'] != 0].drop(columns=['status', 'type', 'railway', 'alt_name', 'old_name', 'official_name', 'user']),
    how='left',
    on='esr'
).drop(columns=['esr']).dropna(subset=['lat', 'lon']).sort_values(by=['name']).reset_index(drop=True)[[
    'name', 'division', 'railway', 'type', 'osm_id', 'lat', 'lon'
]]
moscow_railway_stations.osm_id = moscow_railway_stations.osm_id.apply(lambda x: str(x).split('.')[0])

## Functions

In [25]:
def fill_cadastr_num(character, desc):
    pattern = r"\b\d{2}:\d{2}:\d{6,7}(?::\d{1,4})?(?::\d)?(?::[А-Яа-я\d]*)?\b"

    for char in character:
        if char.get('code') == 'CadastralNumber':
            cad_num = char.get('characteristicValue')
            if cad_num:
                return cad_num.strip() if cad_num != '-' else np.nan
            else:
                matches = re.findall(pattern, desc)
                return matches[0].strip() if matches else np.nan

In [26]:
def fill_area(character):
    for char in character:
        if char.get('code') == 'SquareZU':
            return char.get('characteristicValue')

In [27]:
def get_values_from_rosreestr(cadastral_number: str) -> list:
    try:
        area = Area(
            code=cadastral_number,
            with_proxy=True,
            use_cache=True,
            with_log=False
        )
        coords = list(map(tuple, area.get_coord()[0][0]))
        coords_center = area.get_center_xy()[0][0][0]
        address = area.get_attrs()['address']
        return [coords, coords_center, address]
    except:
        return np.nan

In [28]:
def get_mapbox_isochrones(coords: list, minutes=30) -> Polygon: 
    global mapbox_requests_num

    if not coords:
        return np.nan
    
    if mapbox_requests_num == 300:
        print('Достигли 300 запросов за минуту.\nОжидаем 1 минуту...')
        time.sleep(70)
        
    mapbox_token = "pk.eyJ1IjoiamFidWxhbmk5OTU1IiwiYSI6ImNrejJnaDV0ZTAwZDIyd3FmcmVoejR5bGUifQ.HtEP4kt_8ji1VbaumpXVZg"
    profile = "mapbox/walking" # driving-traffic, driving, cycling
    mapbox_iso_url = rf"https://api.mapbox.com/isochrone/v1/{profile}/{coords[0]},{coords[1]}?contours_minutes={minutes}&polygons=true&access_token={mapbox_token}"
    
    response = requests.get(mapbox_iso_url)
    if response.status_code != 200:
        return np.nan
    mapbox_requests_num += 1
    
    isochrone_data = response.json()
    if 'features' in isochrone_data:
        if len(isochrone_data['features']) == 1:
            return Polygon(map(tuple, isochrone_data['features'][0]['geometry']['coordinates'][0]))
        else:
            return np.nan
    else:
        return np.nan    

In [29]:
def is_railway_near(isochrone_coords):
    if pd.isnull(isochrone_coords):
        return np.nan
    
    polygon = Polygon(isochrone_coords)

    railway_df = moscow_railway_stations.copy()
    railway_df['point'] = railway_df.apply(lambda x: Point(x['lon'], x['lat']), axis=1)

    return any([polygon.contains(point) for point in railway_df['point'].to_list()])

## Main

In [82]:
data_parts = []
mapbox_requests_num = 1

try:
    with open(r'TORGI_Московская область_13_10_2023_11_02_01.json', encoding='utf-8') as f:
        data = pd.read_json(f)

    for i in data['content']:
        data_parts.extend(i)

    df = pd.DataFrame.from_records(data_parts)

    df_columns = [
        'id',
        'lotStatus',
        'biddType',
        'biddForm',
        'lotName',
        'lotDescription',
        'priceMin',
        'createDate',
        'biddEndTime',
        'lotImages',
        'category',
        'characteristics',
    ]
    df = df[df_columns].reset_index(drop=True)
    df['biddEndTime'] = pd.to_datetime(df['biddEndTime']).dt.tz_localize(None)
    df['createDate'] = pd.to_datetime(df['createDate']).dt.tz_localize(None)
    df['biddType'] = df['biddType'].apply(lambda x: x['name'])
    df['biddForm'] = df['biddForm'].apply(lambda x: x['name'])
    df['category'] = df['category'].apply(lambda x: x['name'])
    df['area'] = df['characteristics'].apply(fill_area)
    df['cadastral_number'] = df.apply(lambda x: fill_cadastr_num(x['characteristics'], x['lotDescription']), axis=1)
    
    
    df = df.groupby('cadastral_number')\
        .apply(lambda x: x.loc[x['createDate'].idxmax()])\
        .sort_values(by='createDate', ascending=False)\
        .reset_index(drop=True)
    
    df['lotImages'] = df['lotImages'].apply(lambda x: ', '.join(['https://torgi.gov.ru/new/file-store/v1/'+i+'?disposition=inline' for i in x]))
    df['link'] = df['id'].apply(lambda x: 'https://torgi.gov.ru/new/public/lots/lot/' + x) 
    df['rosreestr_info'] = df['cadastral_number'].apply(get_values_from_rosreestr)
    # df.dropna(subset=['rosreestr_info'], inplace=True)
    df['coords'] = df['rosreestr_info'].apply(lambda x: ', '.join(x[0]))
    # df['coords_center'] = df['rosreestr_info'].apply(lambda x: x[1])
    # df['address'] = df['rosreestr_info'].apply(lambda x: x[2])

    # df.drop(columns=['characteristics', 'rosreestr_info'], inplace=True)

    # df['isochrones'] = df['coords_center'].apply(get_mapbox_isochrones)
    # df['railway_in_30min'] = df.isochrones.apply(is_railway_near)

    # df = df.reset_index(drop=True)

    # df.to_csv(r'TORGI_MSK_FINAL_v0_02.csv', index=False)
except Exception as e:
    print(e)

In [83]:
df[127:130]

,id,lotStatus,biddType,biddForm,lotName,lotDescription,priceMin,createDate,biddEndTime,lotImages,category,characteristics,area,cadastral_number,link,rosreestr_info
127,21000004710000007861_1,APPLICATIONS_SUBMISSION,Аренда и продажа земельных участков,Электронный аукцион,Продажа земельного участка 1 526 кв.м. для ЛПХ...,Продажа земельного участка 1 526 кв.м. для ЛПХ...,1315350.96,2023-10-06 13:08:15.373,2023-11-13 15:00:00,https://torgi.gov.ru/new/file-store/v1/651ff96...,Земли населенных пунктов,"[{'characteristicValue': 1526.0, 'name': 'Площ...",1526.0,50:24:0050401:1938,https://torgi.gov.ru/new/public/lots/lot/21000...,"[[(38.981784654139474, 55.78256690510781), (38..."
128,21000004710000007853_1,APPLICATIONS_SUBMISSION,Аренда и продажа земельных участков,Электронный аукцион,Продажа земельного участка 1 500 кв.м. для ИЖС...,Продажа земельного участка 1 500 кв.м. для ИЖС...,622050.00,2023-10-06 13:08:15.323,2023-11-13 15:00:00,https://torgi.gov.ru/new/file-store/v1/652002c...,Земли населенных пунктов,"[{'characteristicValue': 1500.0, 'name': 'Площ...",1500.0,50:07:0020209:111,https://torgi.gov.ru/new/public/lots/lot/21000...,"[[(35.86442251256344, 56.09638038258705), (35...."
129,21000004710000007856_1,APPLICATIONS_SUBMISSION,Аренда и продажа земельных участков,Электронный аукцион,Аренда земельного участка 900 кв.м. для ЛПХ в ...,Аренда земельного участка 900 кв.м. для ЛПХ в ...,32458.95,2023-10-06 13:08:15.275,2023-11-17 15:00:00,https://torgi.gov.ru/new/file-store/v1/6520052...,Земли населенных пунктов,"[{'characteristicValue': 900.0, 'name': 'Площа...",900.0,50:29:0030303:7921,https://torgi.gov.ru/new/public/lots/lot/21000...,"[[(38.50916358081517, 55.42845959734262), (38...."


In [80]:
df.loc[128, 'lotImages']

'https://torgi.gov.ru/new/file-store/v1/652002c6267f8f472e471cbd?disposition=inline, https://torgi.gov.ru/new/file-store/v1/652002c603519b0e0f5c845c?disposition=inline'

## Load previous df

In [54]:
prev_df = pd.read_csv(r'TORGI_MSK_FINAL_v0_01.csv', index_col=0)

## Построение карты

In [418]:
railway_stations['geometry'] = railway_stations.apply(lambda x: Point((x['lon'], x['lat'])), axis=1)
railway_stations_gpd = gpd.GeoDataFrame(railway_stations[['name', 'geometry']], geometry='geometry', crs='EPSG:4326')

In [419]:
railway_stations_gjson = folium.GeoJson(
    railway_stations_gpd, 
    name="Ж/Д Станции",
    highlight_function=lambda x: {'fillOpacity': 0.8},
    zoom_on_click=True
)

In [16]:
m = folium.Map(
    location=(55.55505165314584, 38.809590793576945), 
    # tiles = 'cartodbpositron', 
    zoom_start=11, 
    control_scale=True
)

railway_stations_gjson.add_to(m)
folium.LayerControl().add_to(m)
m

In [423]:
fol_map = folium.Map(
    location=df_lots_exmaple.loc[4, 'coords_center'][::-1],
    control_scale=True,
    zoom_start=12
)

building = folium.Marker(
    location=df_lots_exmaple.loc[4, 'coords_center'][::-1],
    tooltip="Дом",
    icon=folium.Icon(color="green", icon="glyphicon glyphicon-home")
)
building.add_to(fol_map)

for _, r in df_lots_exmaple[4:].iterrows():
    sim_geo = gpd.GeoSeries(r["isochrones"]).simplify(tolerance=0.001)
    geo_j = sim_geo.to_json()
    geo_j = folium.GeoJson(data=geo_j, style_function=lambda _: {"fillColor": "orange"})
    folium.Popup("30 минут пешком").add_to(geo_j)
    geo_j.add_to(fol_map)


railway_stations_gjson.add_to(fol_map)
folium.LayerControl().add_to(fol_map)

# railway_stations['geometry'] = railway_stations.apply(lambda x: Point((x['lon'], x['lat'])), axis=1)
# railway_stations_gpd = gpd.GeoDataFrame(railway_stations[['name', 'geometry']], geometry='geometry', crs='EPSG:4326')
# railway_stations_gpd["x"] = railway_stations_gpd["geometry"].x
# railway_stations_gpd["y"] = railway_stations_gpd["geometry"].y
# locations = {f"<b>{name}</b>": (y, x) for y, x, name in zip(railway_stations_gpd['y'], railway_stations_gpd['x'], railway_stations_gpd['name'])}
# marker_cluster = MarkerCluster(
#     locations.values(), 
#     popups=list(locations.keys()),
#     name="Ж/Д Станции",
#     )
# marker_cluster.add_to(fol_map)
# # railway_layer = folium.features.GeoJson(
# #     railway_stations_gpd,
# #     name="Ж/Д Станции"
# # )
# # # railway_stations_gpd
# # railway_layer.add_to(fol_map)
# folium.LayerControl().add_to(fol_map)
# for _, r in railway_stations.iterrows():
#     # Without simplifying the representation of each borough,
#     # the map might not be displayed
#     sim_geo = gpd.GeoSeries(Point((r['lon'], r['lat']))).simplify(tolerance=0.001)
#     geo_j = sim_geo.to_json()
#     geo_j = folium.GeoJson(data=geo_j)
#     folium.Marker(f"<b>{r['name']}</b>", max_width=50).add_to(geo_j)
#     geo_j.add_to(fol_map)

fol_map

In [424]:
fol_map.save('example.html')

# TODO
1. Координаты по адресу;
2. Высоту местности;
3. Координаты Ж/Д станций;
4. ...

In [43]:
df_lots_exmaple = df.copy().sample(5, random_state=42).reset_index(drop=True)
df_railway_exmaple = moscow_railway_stations.copy().sample(5, random_state=42).reset_index(drop=True)

In [46]:
df_lots_exmaple.loc[0, 'coords']

'[(38.11404707840167, 56.08222824700549), (38.1138688706156, 56.08203301490809), (38.1142890486066, 56.08189525708516), (38.11449832013519, 56.08209596368307)]'

In [48]:
df_lots_exmaple.loc[0, 'isochrones']

'POLYGON ((38.121179 56.098224, 38.117005 56.097063, 38.117633 56.096063, 38.117179 56.095777, 38.116179 56.096739, 38.11346 56.096343, 38.111179 56.096868, 38.110179 56.096344, 38.109179 56.096878, 38.108304 56.095938, 38.108179 56.092927, 38.107916 56.093799, 38.107026 56.094063, 38.106581 56.095464, 38.103592 56.09565, 38.103179 56.094788, 38.100503 56.093739, 38.100341 56.093063, 38.100333 56.092063, 38.102261 56.091144, 38.103249 56.090133, 38.102264 56.088978, 38.097915 56.088326, 38.097179 56.086273, 38.095179 56.086621, 38.094304 56.085937, 38.092179 56.085829, 38.091179 56.086629, 38.090179 56.086629, 38.089179 56.085574, 38.087601 56.085063, 38.088843 56.084726, 38.089179 56.084033, 38.089836 56.084406, 38.090341 56.084224, 38.090278 56.080063, 38.0899 56.080783, 38.089179 56.080923, 38.088748 56.080494, 38.088551 56.078691, 38.087378 56.078063, 38.087354 56.076888, 38.086507 56.076063, 38.087179 56.075391, 38.088648 56.075593, 38.089055 56.077187, 38.090893 56.077348, 38.091

In [392]:
Polygon(df_lots_exmaple.loc[0, 'isochrones']).contains(Point(df_lots_exmaple.loc[0, 'coords_center']))

True

In [354]:
df_railway_exmaple.head(1)

,division,region,railway,type,esr,status,osm_id,lat,lon,name
0,Московско-Смоленское,Московская область + Москва,Московская,станция,181600,1.0,1.994650e+09,55.744057,37.514675,Фили


In [362]:
df_railway_exmaple['point'] = df_railway_exmaple.apply(lambda x: Point(x['lon'], x['lat']), axis=1)

In [364]:
df_railway_exmaple

,division,region,railway,type,esr,status,osm_id,lat,lon,name,point
0,Московско-Смоленское,Московская область + Москва,Московская,станция,181600,1.0,1.994650e+09,55.744057,37.514675,Фили,POINT (37.51467514 55.7440567)
1,Московско-Смоленское,Московская область + Москва,Московская,остановочный пункт,197327,1.0,1.994610e+09,55.947937,36.613419,Ядрошино,POINT (36.61341858 55.94793701)
2,Московско-Курское,Московская область + Москва,Московская,остановочный пункт,236216,1.0,3.273658e+08,56.283310,38.077717,Семхоз,POINT (38.07771683 56.28330994)
3,Московско-Курское,Московская область + Москва,Московская,остановочный пункт,192821,1.0,2.000569e+09,55.395027,37.805435,Взлётная,POINT (37.80543518 55.39502716)
4,Московско-Смоленское,Московская область + Москва,Московская,остановочный пункт,238813,1.0,3.529458e+08,56.581169,37.567574,94 км,POINT (37.56757355 56.58116913)


In [353]:
df_lots_exmaple.head(1)

,id,lotStatus,biddType,biddForm,lotName,lotDescription,priceMin,createDate,biddEndTime,lotImages,category,area,cadastral_number,link,coords,coords_center,address,isochrones
0,21000004710000002408_1,APPLICATIONS_SUBMISSION,Аренда и продажа земельных участков,Электронный аукцион,Аренда земельного участка 770 кв.м. для ИЖС в ...,Аренда земельного участка 770 кв.м. для ИЖС в ...,241610.6,2022-08-03 11:00:08.666,2023-11-01 15:00:00,[https://torgi.gov.ru/new/file-store/v1/62ea52...,Земли населенных пунктов,770.0,50:17:0011301:695,https://torgi.gov.ru/new/public/lots/lot/21000...,"[(38.69975823644885, 55.86383642715922), (38.6...","[38.69961610574202, 55.86364046234904]","Российская Федерация, Московская область, горо...","POLYGON ((38.701616 55.880981, 38.700306 55.88..."


In [53]:
df_lots_exmaple.loc[0, 'lotImages']

"['https://torgi.gov.ru/new/file-store/v1/64f089735c512c78dfd0ba5b?disposition=inline', 'https://torgi.gov.ru/new/file-store/v1/64f0896b5c512c78dfd0ba37?disposition=inline']"

In [ ]:
ex_map = folium.Map(
    location=df_lots_exmaple.loc[0, 'coords'][0]
)